# 🎙️ Sir C.V. Raman 200Q Marathon — Chatterbox Full Audio Pipeline
Proven 1-click script from `kaggle_voice_cloner.ipynb`!

### Setup:
1. Accelerator: **GPU T4 x2** (Internet: ON)
2. Upload `CV_Raman_200_Human_Script_FINAL.json`
3. Click **Run All**

In [ ]:
# STEP 1: Dependencies & GPU Setup
!pip install -q chatterbox-tts torchvision==0.21.0 yt-dlp "setuptools<81"
import torch, torchvision
torch.backends.cudnn.enabled = False
assert torch.cuda.is_available(), "❌ GPU is OFF! Enable Accelerator -> GPU in Kaggle Settings."
print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)} | torch={torch.__version__} | tv={torchvision.__version__}")


In [ ]:
# STEP 2: Voice Reference
import os, glob, subprocess

clean_ref = "/kaggle/working/clean_voice_ref.wav"
master_refs = glob.glob("/kaggle/input/**/shravan_sir_master_ref.wav", recursive=True) + glob.glob("/kaggle/input/**/intro_0_20.wav", recursive=True)
if master_refs:
    raw_wav = master_refs[0]
    print(f"✅ Found existing dataset voice sample: {raw_wav}")
else:
    print("⏳ Downloading clean voice reference from YouTube...")
    os.makedirs("/kaggle/working/ref", exist_ok=True)
    raw_wav = "/kaggle/working/ref/intro_0_20.wav"
    subprocess.run([
        "yt-dlp", "-f", "ba", "--download-sections", "*0:00-0:20",
        "-x", "--audio-format", "wav",
        "-o", "/kaggle/working/ref/intro_0_20.%(ext)s",
        "https://youtu.be/ixCAZT2vHcM"
    ], check=True)

subprocess.run(["ffmpeg", "-y", "-i", raw_wav, "-ac", "1", "-ar", "24000", "-af", "highpass=f=80,loudnorm=I=-16:TP=-1.5:LRA=11", clean_ref], check=True)
print(f"🎉 Clean Voice Reference Ready: {clean_ref}")


In [ ]:
# STEP 3: Load Chatterbox Multilingual Model
import torch
torch.backends.cudnn.enabled = False
import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
print("✅ Chatterbox Multilingual Model Ready!")


In [ ]:
# STEP 4: Generate All 200 Questions (402 Audios) - 100% Self-Contained!
import os, shutil, json, zlib, base64, time, glob, torchaudio as ta

# Embedded 200 questions data (Zero uploads needed!)
_EMBEDDED_DATA = "..."
questions = json.loads(zlib.decompress(base64.b64decode(_EMBEDDED_DATA)))
print(f"✅ Loaded {len(questions)} questions directly (No file upload required)!")

SCENES = [
    {
        'id': 'speech_intro',
        'text': 'नमस्कार साथियों, मैं श्रवण कुमार सागर। आज की इस विशेष महामैराथन क्लास में आप सभी का स्वागत है। चलिए सीधे प्रश्न संख्या 1 से शुरू करते हैं।'
    }
]
for q in questions:
    num = q['num']
    SCENES.append({'id': f'speech_q_{num:03d}', 'text': q['voice_q']})
    SCENES.append({'id': f'speech_ans_{num:03d}', 'text': q['voice_ans']})

SCENES.append({
    'id': 'speech_outro',
    'text': 'शानदार प्रदर्शन साथियों। पूरे 200 प्रश्न हम लोगों ने लाइव हल कर लिए हैं। आपने जितने भी प्रश्नों का सही उत्तर दिया, अपना कुल स्कोर कमेंट में जरूर बताएं। वीडियो को लाइक करें, अपने दोस्तों के साथ शेयर करें और चैनल को सब्सक्राइब कर लें। आप सभी को परीक्षा के लिए ढेरों शुभकामनाएं। जय हिंद, जय भारत।'
})

OUT_DIR = '/kaggle/working/cv_raman_200_audios'
os.makedirs(OUT_DIR, exist_ok=True)

# Select voice reference
voice_ref = '/kaggle/working/clean_voice_ref.wav'
master_refs = glob.glob('/kaggle/input/**/shravan_sir_master_ref.wav', recursive=True)
if master_refs:
    voice_ref = master_refs[0]

print(f"🎙️ Using voice reference: {voice_ref}")

total = len(SCENES)
start_time = time.time()
print(f"🚀 Generating all {total} clips with Auto-Resume enabled...")

for idx, scene in enumerate(SCENES, 1):
    out_path = os.path.join(OUT_DIR, f"{scene['id']}.wav")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 2000:
        continue
    wav = model.generate(
        text=scene['text'],
        language_id='hi',
        audio_prompt_path=voice_ref,
        exaggeration=0.45,
        cfg_weight=0.55,
        temperature=0.45,
        repetition_penalty=2.0
    )
    ta.save(out_path, wav, model.sr)
    if idx % 10 == 0 or idx == total:
        elapsed = (time.time() - start_time) / 60
        print(f"🎙️ Progress: {idx}/{total} clips generated ({elapsed:.1f} mins elapsed)...")

zip_path = shutil.make_archive('/kaggle/working/cv_raman_200_audios_final', 'zip', OUT_DIR)
print(f"\n📦 ALL DONE! Final zip created at: {zip_path}")
